# 05 - Advanced modelling (LightGBM), tuning and calibration

Reads the tuning and calibration-selection tables written by `scripts/train_model.py`.

> Notebooks display **aggregates only** (no individual loan records) and reuse functions from `src/creditiq/`. Educational project - not for real lending decisions.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from creditiq.config import load_config, setup_logging
from creditiq.cli import processed_path, clean_all_path
from creditiq.reporting import figures as figs

setup_logging("WARNING")
cfg = load_config()
figs.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
TARGET = cfg["target"]["name"]

In [2]:
import json
mdir = cfg.path("models_dir")
for fs in cfg["features"]["sets"]:
    meta = json.loads((mdir / fs / "metadata.json").read_text())
    display(Markdown(f"### Feature set `{fs}` - selected: **{meta['selected_model']}**"))
    display(pd.read_csv(mdir / fs / "tuning_lightgbm.csv"))
    display(pd.read_csv(mdir / fs / "validation_metrics.csv"))
    for m in ("lightgbm", "logistic_regression"):
        print(f"Calibration selection ({m}) - 5-fold CV inside the calibration split:")
        display(pd.read_csv(mdir / fs / f"calibration_selection_{m}.csv"))
    print({m: (v["calibration_method"], round(v["threshold"], 4)) for m, v in meta["models"].items()})

### Feature set `applicant` - selected: **lightgbm**

,num_leaves,min_child_samples,reg_lambda,best_iteration,val_roc_auc,val_log_loss,selected
0,15,200,1.0,253,0.674602,0.421132,False
1,31,200,1.0,191,0.675392,0.420969,False
2,63,500,5.0,128,0.675327,0.420894,True
3,31,100,0.0,193,0.674503,0.421182,False


,model,scores,n,prevalence,mean_predicted,roc_auc,pr_auc,brier,log_loss,ece,fit_seconds
0,dummy,raw,47013,0.164869,0.157665,0.500000,0.164869,0.137739,0.447850,0.007204,0.779306
1,logistic_regression,raw,47013,0.164869,0.161117,0.669832,0.277988,0.130680,0.422794,0.005610,10.691067
2,lightgbm,raw,47013,0.164869,0.160579,0.675327,0.283555,0.130162,0.420894,0.004311,31.451409


Calibration selection (lightgbm) - 5-fold CV inside the calibration split:


,method,cv_brier,cv_brier_std,cv_log_loss,cv_log_loss_std,selected
0,none,0.140558,0.000461,0.444383,0.001749,False
1,sigmoid,0.139554,0.000566,0.441785,0.001942,True
2,isotonic,0.139603,0.000618,0.442055,0.002108,False


...

Calibration selection (lightgbm) - 5-fold CV inside the calibration split:


,method,cv_brier,cv_brier_std,cv_log_loss,cv_log_loss_std,selected
0,none,0.139350,0.000560,0.440110,0.002102,False
1,sigmoid,0.138046,0.000684,0.436714,0.002316,True
2,isotonic,0.138087,0.000719,0.437107,0.002705,False


Calibration selection (logistic_regression) - 5-fold CV inside the calibration split:


,method,cv_brier,cv_brier_std,cv_log_loss,cv_log_loss_std,selected
0,none,0.140539,0.000477,0.444060,0.001913,False
1,sigmoid,0.138791,0.000614,0.438964,0.002109,False
2,isotonic,0.138785,0.000594,0.439273,0.002597,True


{'dummy': ('none', 0.5), 'logistic_regression': ('isotonic', 0.224), 'lightgbm': ('sigmoid', 0.2039)}


**Design notes**

* Bounded grid (4 LightGBM configurations x early stopping on the validation split).
* No class re-weighting for LightGBM: it would distort probabilities; imbalance is handled by
  evaluating PR-AUC and by threshold choice. Logistic regression tries `class_weight` in its grid.
* Calibration is fitted on 2014 Q2 (a split never used for fitting or model selection) and the
  method is chosen by cross-validation *inside* that split, so the test set stays untouched.